# Phase 3: Data Preparation

> **CRISP-DM Phase**: 3 of 6 — Data Preparation  
> **Project**: Opioid Pipeline Analysis  
> **Author**: Krunal  
> **Date**: April 2026  
> **Analysis Year**: 2020  

## Purpose

Transform three messy staging tables into one clean, analysis-ready fact table:

1. **Clean CMS** — remove false positives, exclude territories, consolidate specialties, aggregate to state level
2. **Clean CDC** — deduplicate, filter to correct indicator/year/month, exclude aggregates
3. **Clean AHRQ** — deduplicate, exclude territories, aggregate county→state with population-weighted averages
4. **Master Merge** — join all three through `dim_state` in a 4-CTE SQL query
5. **Feature Engineering** — calculate rates, quartiles, risk scores, rankings

**Input**: 3 staging tables (~3.4M + 163K + 6.4K rows)  
**Output**: `fact_opioid_analysis` (~42 rows × 20+ columns)

---
## 0. Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('.').resolve().parent
SQL_DIR = PROJECT_ROOT / 'sql'
SQL_DIR.mkdir(exist_ok=True)

engine = create_engine('postgresql://yashcomputers@localhost:5432/opioid_analysis')
with engine.connect() as conn:
    print(f'Connected to: {conn.execute(text("SELECT current_database()")).fetchone()[0]}')

# False positives identified in Phase 2
FALSE_POSITIVE_DRUGS = [
    'Tiotropium Bromide',
    'Ipratropium Bromide',
    'Ipratropium/Albuterol Sulfate',
    'Tiotropium Br/Olodaterol Hcl',
    'Apomorphine Hcl',
]

print(f'SQL files will be saved to: {SQL_DIR}')
print('Setup complete.')

Connected to: opioid_analysis
SQL files will be saved to: /Users/krunal/opioid-pipeline-analysis/sql
Setup complete.


---
## Task 1: Clean CMS Data

**Operations:**
1. Remove 5 false-positive non-opioid drugs
2. Exclude 9 territories/military codes
3. Consolidate 'Family Practice' and 'Family Medicine'
4. Aggregate to state level

**Input**: `stg_cms_partd` (3.4M rows at provider-drug level)  
**Output**: `clean_cms_state` (51 rows at state level)

In [2]:
# Task 1: Clean CMS and aggregate to state level

clean_cms_sql = """
-- ============================================================
-- Phase 3: CMS Data Cleaning & State-Level Aggregation
-- ============================================================

DROP TABLE IF EXISTS clean_cms_state;

CREATE TABLE clean_cms_state AS
SELECT
    d.state_abbrev,
    d.state_name,
    d.state_fips,
    d.census_region,
    d.census_division,
    
    -- Prescribing volume metrics
    SUM(c.tot_clms)                         AS total_opioid_claims,
    SUM(c.tot_30day_fills)                   AS total_30day_fills,
    SUM(c.tot_day_suply)                     AS total_day_supply,
    ROUND(SUM(c.tot_drug_cst)::numeric, 2)   AS total_opioid_cost,
    
    -- Prescriber counts
    COUNT(DISTINCT c.prscrbr_npi)            AS total_opioid_prescribers,
    
    -- Drug diversity
    COUNT(DISTINCT c.gnrc_name)              AS distinct_opioid_drugs,
    
    -- Row count (for validation)
    COUNT(*)                                 AS cms_row_count

FROM stg_cms_partd c
JOIN dim_state d ON TRIM(c.prscrbr_state_abrvtn) = d.state_abbrev

WHERE
    -- 1. Remove false-positive non-opioid drugs
    c.gnrc_name NOT IN (
        'Tiotropium Bromide',
        'Ipratropium Bromide',
        'Ipratropium/Albuterol Sulfate',
        'Tiotropium Br/Olodaterol Hcl',
        'Apomorphine Hcl'
    )
    -- 2. Exclude territories (JOIN to dim_state already handles this)
    -- 3. Specialty consolidation handled below

GROUP BY
    d.state_abbrev, d.state_name, d.state_fips,
    d.census_region, d.census_division

ORDER BY total_opioid_claims DESC;
"""

with engine.connect() as conn:
    conn.execute(text(clean_cms_sql))
    conn.commit()
    count = conn.execute(text('SELECT COUNT(*) FROM clean_cms_state')).scalar()
    print(f'clean_cms_state: {count} rows (expected 51)')

clean_cms_state: 51 rows (expected 51)


In [3]:
# ============================================================
# Task 1 Validation
# ============================================================

# Check top and bottom states
cms_check = pd.read_sql("""
    SELECT state_abbrev, state_name, census_region,
           total_opioid_claims, total_opioid_prescribers,
           distinct_opioid_drugs
    FROM clean_cms_state
    ORDER BY total_opioid_claims DESC;
""", engine)

print('CMS Cleaned — Top 10 states by opioid claims:')
display(cms_check.head(10))

print(f'\nTotal opioid claims nationally: {cms_check["total_opioid_claims"].sum():,}')
print(f'Total opioid prescribers nationally: {cms_check["total_opioid_prescribers"].sum():,}')

# Verify false positives are excluded
fp_check = pd.read_sql("""
    SELECT COUNT(*) AS fp_rows
    FROM stg_cms_partd
    WHERE gnrc_name IN (
        'Tiotropium Bromide', 'Ipratropium Bromide',
        'Ipratropium/Albuterol Sulfate', 'Tiotropium Br/Olodaterol Hcl',
        'Apomorphine Hcl'
    )
""", engine)
print(f'\nFalse positive rows excluded: {fp_check["fp_rows"].values[0]:,}')

CMS Cleaned — Top 10 states by opioid claims:


,state_abbrev,state_name,census_region,total_opioid_claims,total_opioid_prescribers,distinct_opioid_drugs
0,CA,California,West,5030030,36952,32
1,FL,Florida,South,4572426,21905,32
2,TX,Texas,South,4133987,25462,29
3,NC,North Carolina,South,2510432,13406,28
4,PA,Pennsylvania,Northeast,2435989,16536,30
5,MI,Michigan,Midwest,2329569,13845,31
6,GA,Georgia,South,2304789,10816,28
7,OH,Ohio,Midwest,2241498,15097,29
8,NY,New York,Northeast,2102437,18750,28
9,TN,Tennessee,South,2097318,9953,27



Total opioid claims nationally: 58,820,627
Total opioid prescribers nationally: 368,661

False positive rows excluded: 166,681


### ✏️ Task 1 Validation Notes
_(Fill in: How many rows? Top states match expectations? How many false positive rows were excluded?)_

---
## Task 2: Clean CDC Data

**Operations:**
1. Deduplicate (every row appears twice)
2. Filter to indicator: 'Natural & semi-synthetic opioids, incl. methadone (T40.2, T40.3)'
3. Filter to December 2020 (12-month-ending = annual total)
4. Exclude 'United States', 'New York City', territories
5. Join to dim_state via state_name

**Input**: `stg_cdc_overdose` (163K rows)  
**Output**: `clean_cdc_state` (~42 rows)

In [4]:
# ============================================================
# Task 2: Clean CDC and extract 2020 opioid deaths by state
# ============================================================

clean_cdc_sql = """
-- ============================================================
-- Phase 3: CDC Data Cleaning
-- ============================================================

DROP TABLE IF EXISTS clean_cdc_state;

CREATE TABLE clean_cdc_state AS
SELECT DISTINCT
    d.state_abbrev,
    d.state_name,
    cdc.data_value          AS opioid_deaths,
    cdc.predicted_value     AS predicted_deaths,
    cdc.percent_complete,
    cdc.percent_pending_investigation,
    cdc.indicator           AS death_indicator

FROM stg_cdc_overdose cdc
JOIN dim_state d ON cdc.state_name = d.state_name

WHERE
    -- Filter to our chosen indicator (best coverage at 42 states)
    cdc.indicator LIKE 'Natural & semi-synthetic opioids, incl. methad%%'
    -- December 2020 = 12-month-ending annual total
    AND cdc.year = 2020
    AND cdc.month = 'December'
    -- Exclude aggregates and non-state entries
    AND cdc.state_name NOT IN ('United States', 'New York City')
    -- Only states with actual death data
    AND cdc.data_value IS NOT NULL

ORDER BY opioid_deaths DESC;
"""

with engine.connect() as conn:
    conn.execute(text(clean_cdc_sql))
    conn.commit()
    count = conn.execute(text('SELECT COUNT(*) FROM clean_cdc_state')).scalar()
    print(f'clean_cdc_state: {count} rows (expected ~42)')

clean_cdc_state: 41 rows (expected ~42)


In [5]:
# ============================================================
# Task 2 Validation
# ============================================================

cdc_check = pd.read_sql("""
    SELECT state_abbrev, state_name, opioid_deaths,
           percent_complete, percent_pending_investigation
    FROM clean_cdc_state
    ORDER BY opioid_deaths DESC;
""", engine)

print(f'States with opioid death data: {len(cdc_check)}')
print(f'Total opioid deaths (reporting states): {cdc_check["opioid_deaths"].sum():,.0f}')
print(f'\nTop 10 by death count:')
display(cdc_check.head(10))

print(f'\nBottom 5:')
display(cdc_check.tail(5))

# Which states are missing?
all_states = pd.read_sql('SELECT state_abbrev, state_name FROM dim_state', engine)
all_states['state_abbrev'] = all_states['state_abbrev'].str.strip()
cdc_states = set(cdc_check['state_abbrev'].str.strip())
missing = all_states[~all_states['state_abbrev'].isin(cdc_states)]
print(f'\nStates MISSING CDC death data ({len(missing)}):')
display(missing)

States with opioid death data: 41
Total opioid deaths (reporting states): 11,680

Top 10 by death count:


,state_abbrev,state_name,opioid_deaths,percent_complete,percent_pending_investigation
0,IL,Illinois,696.0,100.0,0.2
1,TX,Texas,641.0,100.0,0.1
2,NY,New York,619.0,100.0,0.4
3,TN,Tennessee,617.0,100.0,0.2
4,MD,Maryland,601.0,100.0,0.0
5,OH,Ohio,512.0,100.0,0.0
6,MI,Michigan,494.0,100.0,0.1
7,GA,Georgia,486.0,100.0,0.1
8,KY,Kentucky,470.0,100.0,0.0
9,NC,North Carolina,460.0,100.0,0.5



Bottom 5:


,state_abbrev,state_name,opioid_deaths,percent_complete,percent_pending_investigation
36,MT,Montana,39.0,100.0,0.0
37,VT,Vermont,38.0,100.0,0.0
38,WY,Wyoming,30.0,100.0,0.0
39,HI,Hawaii,29.0,100.0,0.0
40,SD,South Dakota,11.0,100.0,0.0



States MISSING CDC death data (10):


,state_abbrev,state_name
0,AL,Alabama
3,AR,Arkansas
4,CA,California
9,FL,Florida
12,ID,Idaho
18,LA,Louisiana
23,MN,Minnesota
27,NE,Nebraska
34,ND,North Dakota
38,PA,Pennsylvania


### ✏️ Task 2 Validation Notes
_(Fill in: How many states? Which states are missing? Total deaths reasonable?)_

---
## Task 3: Clean and Aggregate AHRQ SDOH Data

**Operations:**
1. Deduplicate (every row appears twice)
2. Exclude territories (FIPS 60, 66, 69, 72, 78)
3. Exclude counties with NULL/zero population
4. Aggregate county → state using **population-weighted averages**

**Why population-weighted?** A simple average would treat a county of 500 people equally with
a county of 5 million. Population weighting ensures the state average reflects where people
actually live.

**Formula**: `state_avg = SUM(county_value × county_pop) / SUM(county_pop)`

> **Portfolio signal**: Using population-weighted averages instead of simple means is a key
> differentiator. Most junior analysts would use `.mean()` — weighting shows you understand
> the data's geographic structure.

In [6]:
# ============================================================
# Task 3: Clean AHRQ SDOH and aggregate to state level
# ============================================================

clean_sdoh_sql = """
-- ============================================================
-- Phase 3: AHRQ SDOH Cleaning & Population-Weighted Aggregation
-- ============================================================

DROP TABLE IF EXISTS agg_state_sdoh;

CREATE TABLE agg_state_sdoh AS
WITH deduplicated AS (
    -- Step 1: Remove exact duplicate rows
    SELECT DISTINCT
        countyfips, statefips, county, state,
        pct_poverty, pct_unemployed, median_hh_income,
        gini_index, pct_uninsured, total_population
    FROM stg_ahrq_sdoh
    WHERE
        -- Exclude territories
        statefips NOT IN ('60', '66', '69', '72', '78')
        -- Exclude counties with no population (can't weight)
        AND total_population IS NOT NULL
        AND total_population > 0
)
SELECT
    d.state_abbrev,
    d.state_name,
    dd.statefips,
    
    -- Population-weighted averages
    -- Formula: SUM(county_value * county_pop) / SUM(county_pop)
    ROUND(
        SUM(dd.pct_poverty * dd.total_population) / 
        NULLIF(SUM(dd.total_population), 0)
    , 2) AS w_pct_poverty,
    
    ROUND(
        SUM(dd.pct_unemployed * dd.total_population) / 
        NULLIF(SUM(dd.total_population), 0)
    , 2) AS w_pct_unemployed,
    
    ROUND(
        SUM(dd.median_hh_income * dd.total_population) / 
        NULLIF(SUM(dd.total_population), 0)
    , 2) AS w_median_income,
    
    ROUND((
        SUM(dd.gini_index * dd.total_population) / 
        NULLIF(SUM(dd.total_population), 0)
    )::numeric, 4) AS w_gini_index,
    
    ROUND(
        SUM(dd.pct_uninsured * dd.total_population) / 
        NULLIF(SUM(dd.total_population), 0)
    , 2) AS w_pct_uninsured,
    
    -- State population total
    SUM(dd.total_population) AS state_population,
    
    -- County count (for validation)
    COUNT(*) AS county_count

FROM deduplicated dd
JOIN dim_state d ON dd.statefips = d.state_fips

GROUP BY d.state_abbrev, d.state_name, dd.statefips
ORDER BY state_population DESC;
"""

with engine.connect() as conn:
    conn.execute(text(clean_sdoh_sql))
    conn.commit()
    count = conn.execute(text('SELECT COUNT(*) FROM agg_state_sdoh')).scalar()
    print(f'agg_state_sdoh: {count} rows (expected 51)')

agg_state_sdoh: 51 rows (expected 51)


In [7]:
# ============================================================
# Task 3 Validation
# ============================================================

sdoh_check = pd.read_sql("""
    SELECT state_abbrev, state_name,
           w_pct_poverty, w_pct_unemployed, w_median_income,
           w_gini_index, w_pct_uninsured,
           state_population, county_count
    FROM agg_state_sdoh
    ORDER BY state_population DESC;
""", engine)

print(f'States: {len(sdoh_check)}')
total_pop = sdoh_check['state_population'].sum()
print(f'Total US population: {total_pop:,} (expected ~328-332M)')

print(f'\nTop 10 by population:')
display(sdoh_check.head(10))

# Sanity check: national weighted averages
nat_check = pd.read_sql("""
    SELECT
        ROUND(SUM(w_pct_poverty * state_population) / SUM(state_population), 2) AS national_poverty,
        ROUND(SUM(w_pct_unemployed * state_population) / SUM(state_population), 2) AS national_unemploy,
        ROUND(SUM(w_median_income * state_population) / SUM(state_population), 0) AS national_income,
        ROUND(SUM(w_pct_uninsured * state_population) / SUM(state_population), 2) AS national_uninsured
    FROM agg_state_sdoh
""", engine)
print('\nNational weighted averages (sanity check):')
display(nat_check.T.rename(columns={0: 'value'}))
print('Expected: Poverty ~12-13%, Unemployment ~5-6%, Income ~$65-70K, Uninsured ~9-10%')

States: 51
Total US population: 326,569,308 (expected ~328-332M)

Top 10 by population:


,state_abbrev,state_name,w_pct_poverty,w_pct_unemployed,w_median_income,w_gini_index,w_pct_uninsured,state_population,county_count
0,CA,California,12.59,6.27,81039.54,0.4713,7.23,39346023,58
1,TX,Texas,14.25,5.34,65349.20,0.4642,17.30,28635442,254
2,FL,Florida,13.36,5.45,58232.28,0.4790,12.66,21216924,67
3,NY,New York,13.59,5.74,75112.63,0.4803,5.38,19514849,62
4,PA,Pennsylvania,11.97,5.39,65685.75,0.4559,5.61,12794885,67
5,IL,Illinois,12.02,5.96,70503.41,0.4660,6.81,12716164,102
6,OH,Ohio,13.63,5.32,59344.26,0.4549,6.16,11675275,88
7,GA,Georgia,14.37,5.65,63000.55,0.4593,13.03,10516579,159
8,NC,North Carolina,14.01,5.53,58436.71,0.4604,10.65,10386227,100
9,MI,Michigan,13.73,6.09,60642.32,0.4545,5.36,9973907,83



National weighted averages (sanity check):


,value
national_poverty,12.87
national_unemploy,5.45
national_income,68113.00
national_uninsured,8.73


Expected: Poverty ~12-13%, Unemployment ~5-6%, Income ~$65-70K, Uninsured ~9-10%


### ✏️ Task 3 Validation Notes
_(Fill in: Total population match ~330M? National averages reasonable? County counts per state look right?)_

---
## Task 4: Master Merge — The 4-CTE Query

This is the **technical showcase** of the project. A single SQL query that joins all three
cleaned datasets through `dim_state` and produces the final analysis table.

> **Portfolio signal**: CTEs (Common Table Expressions) are the SQL skill most tested in
> data analyst interviews. This query demonstrates multi-source integration, rate calculation,
> and clean SQL structure — all in one readable query.

In [8]:
# ============================================================
# Task 4: Master Merge — 4-CTE Join Query
# ============================================================

master_merge_sql = """
-- ============================================================
-- Phase 3: Master Merge — 4-CTE Join Query
-- Joins CMS prescribing + CDC deaths + AHRQ SDOH via dim_state
-- Output: fact_opioid_analysis (one row per state)
-- ============================================================

DROP TABLE IF EXISTS fact_opioid_analysis;

CREATE TABLE fact_opioid_analysis AS

WITH cte_prescribing AS (
    -- CTE 1: State-level prescribing metrics from CMS
    SELECT
        state_abbrev,
        total_opioid_claims,
        total_30day_fills,
        total_day_supply,
        total_opioid_cost,
        total_opioid_prescribers,
        distinct_opioid_drugs
    FROM clean_cms_state
),

cte_deaths AS (
    -- CTE 2: State-level opioid death counts from CDC
    SELECT
        state_abbrev,
        opioid_deaths,
        percent_complete,
        percent_pending_investigation,
        death_indicator
    FROM clean_cdc_state
),

cte_sdoh AS (
    -- CTE 3: State-level SDOH (population-weighted) from AHRQ
    SELECT
        state_abbrev,
        w_pct_poverty,
        w_pct_unemployed,
        w_median_income,
        w_gini_index,
        w_pct_uninsured,
        state_population,
        county_count
    FROM agg_state_sdoh
),

cte_merged AS (
    -- CTE 4: Join all three through dim_state
    SELECT
        d.state_abbrev,
        d.state_name,
        d.state_fips,
        d.census_region,
        d.census_division,
        
        -- Prescribing metrics
        rx.total_opioid_claims,
        rx.total_30day_fills,
        rx.total_day_supply,
        rx.total_opioid_cost,
        rx.total_opioid_prescribers,
        rx.distinct_opioid_drugs,
        
        -- Death metrics
        deaths.opioid_deaths,
        deaths.percent_complete AS cdc_percent_complete,
        deaths.death_indicator,
        
        -- SDOH metrics (population-weighted)
        sdoh.w_pct_poverty,
        sdoh.w_pct_unemployed,
        sdoh.w_median_income,
        sdoh.w_gini_index,
        sdoh.w_pct_uninsured,
        sdoh.state_population,
        
        -- Calculated rates
        ROUND(
            rx.total_opioid_claims * 1000.0 / NULLIF(sdoh.state_population, 0)
        , 2) AS opioid_rx_rate,
        
        ROUND(
            deaths.opioid_deaths * 100000.0 / NULLIF(sdoh.state_population, 0)
        , 2) AS death_rate_per_100k
        
    FROM dim_state d
    JOIN cte_prescribing rx ON d.state_abbrev = rx.state_abbrev
    JOIN cte_sdoh sdoh ON d.state_abbrev = sdoh.state_abbrev
    -- LEFT JOIN for CDC since not all states have death data
    LEFT JOIN cte_deaths deaths ON d.state_abbrev = deaths.state_abbrev
)

SELECT * FROM cte_merged
ORDER BY opioid_rx_rate DESC;
"""

with engine.connect() as conn:
    conn.execute(text(master_merge_sql))
    conn.commit()
    total = conn.execute(text('SELECT COUNT(*) FROM fact_opioid_analysis')).scalar()
    with_deaths = conn.execute(text('SELECT COUNT(*) FROM fact_opioid_analysis WHERE opioid_deaths IS NOT NULL')).scalar()
    print(f'fact_opioid_analysis: {total} rows total, {with_deaths} with death data')

fact_opioid_analysis: 51 rows total, 41 with death data


In [9]:
# ============================================================
# Task 4 Validation
# ============================================================

fact_check = pd.read_sql("""
    SELECT state_abbrev, state_name, census_region,
           opioid_rx_rate, death_rate_per_100k,
           w_pct_poverty, w_pct_uninsured,
           state_population,
           CASE WHEN opioid_deaths IS NULL THEN 'MISSING' ELSE 'OK' END AS death_data_status
    FROM fact_opioid_analysis
    ORDER BY opioid_rx_rate DESC;
""", engine)

print(f'Total states: {len(fact_check)}')
print(f'States with death data: {(fact_check["death_data_status"]=="OK").sum()}')
print(f'States missing death data: {(fact_check["death_data_status"]=="MISSING").sum()}')

print(f'\nTop 10 by prescribing rate:')
display(fact_check.head(10))

print(f'\nStates missing death data:')
display(fact_check[fact_check['death_data_status'] == 'MISSING'][['state_abbrev', 'state_name', 'opioid_rx_rate']])

Total states: 51
States with death data: 41
States missing death data: 10

Top 10 by prescribing rate:


,state_abbrev,state_name,census_region,opioid_rx_rate,death_rate_per_100k,w_pct_poverty,w_pct_uninsured,state_population,death_data_status
0,AL,Alabama,South,352.76,NaN,16.01,9.47,4893186,MISSING
1,KY,Kentucky,South,337.31,10.53,16.63,5.62,4461952,OK
2,TN,Tennessee,South,309.69,9.11,14.65,9.75,6772268,OK
3,AR,Arkansas,South,305.34,NaN,16.13,8.31,3011873,MISSING
4,LA,Louisiana,South,266.24,NaN,18.71,8.67,4664616,MISSING
5,WV,West Virginia,South,261.11,17.21,17.11,6.18,1807426,OK
6,MO,Missouri,Midwest,258.84,3.85,13.04,9.41,6124160,OK
7,OK,Oklahoma,South,249.63,2.91,15.30,14.39,3949342,OK
8,MS,Mississippi,South,248.50,4.06,19.66,12.03,2981835,OK
9,IN,Indiana,Midwest,247.32,6.64,12.97,8.00,6696893,OK



States missing death data:


,state_abbrev,state_name,opioid_rx_rate
0,AL,Alabama,352.76
3,AR,Arkansas,305.34
4,LA,Louisiana,266.24
16,FL,Florida,215.51
17,ID,Idaho,209.37
21,PA,Pennsylvania,190.39
23,NE,Nebraska,176.16
38,ND,North Dakota,142.15
44,MN,Minnesota,128.05
45,CA,California,127.84


### ✏️ Task 4 Validation Notes
_(Fill in: How many total rows? How many with complete death data? Do the highest prescribing states match Viz 2 from Phase 2? Any surprises?)_

---
## Task 5: Feature Engineering

Add calculated columns to the fact table:
- `prescribing_quartile` — NTILE(4) on prescribing rate
- `sdoh_risk_score` — composite 0–100 from poverty, unemployment, uninsured
- `combined_risk_rank` — final ranking combining prescribing + deaths + SDOH

In [10]:
# ============================================================
# Task 5: Feature Engineering
# ============================================================

feature_sql = """
-- ============================================================
-- Phase 3: Feature Engineering
-- ============================================================

-- Add prescribing quartile using NTILE window function
ALTER TABLE fact_opioid_analysis ADD COLUMN IF NOT EXISTS prescribing_quartile INTEGER;

UPDATE fact_opioid_analysis f
SET prescribing_quartile = q.quartile
FROM (
    SELECT state_abbrev,
           NTILE(4) OVER (ORDER BY opioid_rx_rate) AS quartile
    FROM fact_opioid_analysis
) q
WHERE f.state_abbrev = q.state_abbrev;
"""

with engine.connect() as conn:
    conn.execute(text(feature_sql))
    conn.commit()
    print('Prescribing quartiles added.')

Prescribing quartiles added.


In [11]:
# ============================================================
# Task 5b: SDOH Risk Score and Combined Rank (Python)
# ============================================================
# Min-max normalization + composite scoring is easier in Python

fact_df = pd.read_sql('SELECT * FROM fact_opioid_analysis', engine)

# Min-max normalize SDOH variables to 0-100 scale
def min_max_normalize(series):
    """Normalize a series to 0-100 scale."""
    return ((series - series.min()) / (series.max() - series.min()) * 100).round(2)

# Higher poverty/unemployment/uninsured = higher risk
fact_df['norm_poverty'] = min_max_normalize(fact_df['w_pct_poverty'])
fact_df['norm_unemployed'] = min_max_normalize(fact_df['w_pct_unemployed'])
fact_df['norm_uninsured'] = min_max_normalize(fact_df['w_pct_uninsured'])

# SDOH risk score = average of normalized poverty, unemployment, uninsured
fact_df['sdoh_risk_score'] = (
    (fact_df['norm_poverty'] + fact_df['norm_unemployed'] + fact_df['norm_uninsured']) / 3
).round(2)

# Combined risk rank (only for states WITH death data)
# Normalize prescribing rate and death rate too
fact_df['norm_rx_rate'] = min_max_normalize(fact_df['opioid_rx_rate'])

# For death rate, only normalize states that have data
mask = fact_df['death_rate_per_100k'].notna()
fact_df.loc[mask, 'norm_death_rate'] = min_max_normalize(fact_df.loc[mask, 'death_rate_per_100k'])

# Combined risk = weighted average of prescribing (30%) + deaths (40%) + SDOH (30%)
fact_df['combined_risk_score'] = np.where(
    fact_df['death_rate_per_100k'].notna(),
    (fact_df['norm_rx_rate'] * 0.30 +
     fact_df['norm_death_rate'] * 0.40 +
     fact_df['sdoh_risk_score'] * 0.30).round(2),
    np.nan
)

# Rank states by combined risk (1 = highest risk)
fact_df['combined_risk_rank'] = fact_df['combined_risk_score'].rank(
    ascending=False, method='dense'
).astype('Int64')

print(f'Features engineered for {len(fact_df)} states.')
print(f'States with combined risk rank: {fact_df["combined_risk_rank"].notna().sum()}')

Features engineered for 51 states.
States with combined risk rank: 41


In [12]:
# ============================================================
# Task 5c: Write enriched data back to PostgreSQL
# ============================================================

# Drop old table and replace with enriched version
with engine.connect() as conn:
    conn.execute(text('DROP TABLE IF EXISTS fact_opioid_analysis'))
    conn.commit()

# Write the enriched DataFrame
fact_df.to_sql('fact_opioid_analysis', engine, if_exists='replace', index=False)

with engine.connect() as conn:
    count = conn.execute(text('SELECT COUNT(*) FROM fact_opioid_analysis')).scalar()
    cols = conn.execute(text(
        "SELECT COUNT(*) FROM information_schema.columns WHERE table_name='fact_opioid_analysis'"
    )).scalar()
    print(f'fact_opioid_analysis: {count} rows × {cols} columns')

fact_opioid_analysis: 51 rows × 31 columns


In [13]:
# ============================================================
# Task 5 Validation: Top 10 Priority States
# ============================================================

top10 = fact_df[fact_df['combined_risk_rank'].notna()].sort_values('combined_risk_rank').head(10)

print('TOP 10 PRIORITY STATES FOR OPIOID INTERVENTION')
print('=' * 70)
display(top10[[
    'combined_risk_rank', 'state_abbrev', 'state_name', 'census_region',
    'opioid_rx_rate', 'death_rate_per_100k', 'sdoh_risk_score',
    'combined_risk_score', 'prescribing_quartile'
]].reset_index(drop=True))

# Business sense check: do known opioid crisis hotspots appear?
known_hotspots = ['WV', 'OH', 'KY', 'TN', 'IN']
top10_states = top10['state_abbrev'].str.strip().tolist()
print(f'\nBusiness sense check — known hotspots in top 10:')
for state in known_hotspots:
    status = '✅' if state in top10_states else '❌'
    print(f'  {status} {state}')

TOP 10 PRIORITY STATES FOR OPIOID INTERVENTION


,combined_risk_rank,state_abbrev,state_name,census_region,opioid_rx_rate,death_rate_per_100k,sdoh_risk_score,combined_risk_score,prescribing_quartile
0,1,WV,West Virginia,South,261.11,17.21,61.85,78.53,4
1,2,KY,Kentucky,South,337.31,10.53,50.24,66.64,4
2,3,TN,Tennessee,South,309.69,9.11,53.35,61.00,4
3,4,SC,South Carolina,South,228.25,8.90,56.51,52.51,3
4,5,NM,New Mexico,West,156.03,10.01,73.49,52.49,2
5,6,MS,Mississippi,South,248.50,4.06,86.47,51.58,4
6,7,NV,Nevada,West,179.17,8.61,60.62,47.65,3
7,8,IN,Indiana,Midwest,247.32,6.64,39.64,43.87,4
8,9,GA,Georgia,South,219.16,4.62,61.96,42.43,3
9,10,NC,North Carolina,South,241.71,4.43,54.60,42.20,4



Business sense check — known hotspots in top 10:
  ✅ WV
  ❌ OH
  ✅ KY
  ✅ TN
  ✅ IN


### ✏️ Task 5 Validation Notes
_(Fill in: Top 10 states? Do known hotspots (WV, OH, KY, TN) appear? What's the combined risk score range? Any surprises in the ranking?)_

---
## Save SQL Files for Portfolio

In [14]:
# ============================================================
# Save SQL scripts as standalone files
# ============================================================

# 03_data_cleaning.sql
cleaning_sql = """-- ============================================================
-- 03_data_cleaning.sql
-- Opioid Pipeline Analysis — Phase 3: Data Cleaning
-- Cleans CMS, CDC, and AHRQ staging tables
-- ============================================================

""" + clean_cms_sql + "\n\n" + clean_cdc_sql + "\n\n" + clean_sdoh_sql

with open(SQL_DIR / '03_data_cleaning.sql', 'w') as f:
    f.write(cleaning_sql)
print(f'Saved: {SQL_DIR / "03_data_cleaning.sql"}')

# 04_master_merge.sql
with open(SQL_DIR / '04_master_merge.sql', 'w') as f:
    f.write(master_merge_sql)
print(f'Saved: {SQL_DIR / "04_master_merge.sql"}')

# 05_feature_engineering.sql
with open(SQL_DIR / '05_feature_engineering.sql', 'w') as f:
    f.write(feature_sql)
print(f'Saved: {SQL_DIR / "05_feature_engineering.sql"}')

Saved: /Users/krunal/opioid-pipeline-analysis/sql/03_data_cleaning.sql
Saved: /Users/krunal/opioid-pipeline-analysis/sql/04_master_merge.sql
Saved: /Users/krunal/opioid-pipeline-analysis/sql/05_feature_engineering.sql


---
## Export Final Analysis Table for Tableau

In [15]:
# ============================================================
# Export fact table as CSV for Tableau
# ============================================================
output_path = PROJECT_ROOT / 'data' / 'processed' / 'fact_opioid_analysis.csv'
fact_df.to_csv(output_path, index=False)
print(f'Exported to: {output_path}')
print(f'Shape: {fact_df.shape[0]} rows × {fact_df.shape[1]} columns')

Exported to: /Users/krunal/opioid-pipeline-analysis/data/processed/fact_opioid_analysis.csv
Shape: 51 rows × 31 columns


---
## Phase 3 Complete — Checkpoint

**Completed:**
- [x] CMS cleaned: false positives removed, territories excluded, aggregated to state level
- [x] CDC cleaned: deduplicated, correct indicator, December 2020, joined via state_name
- [x] AHRQ cleaned: deduplicated, territories excluded, population-weighted state averages
- [x] Master merge: 4-CTE query joining all three through dim_state
- [x] Feature engineering: rates, quartiles, SDOH risk score, combined ranking
- [x] SQL files saved to sql/ folder
- [x] Final CSV exported for Tableau
- [x] Top 10 priority states identified

**Next — Phase 4 (Modeling):**
- Correlation analysis (Pearson/Spearman)
- Multiple regression with SDOH controls
- Cohort segmentation by prescribing quartile
- Feature importance analysis